# 手撕 Warmup + Cosine Learning Rate Scheduler

## 背景
大模型标配：warmup 阶段线性增长（避免早期发散），之后 cosine 衰减到 min_lr。
公式：warmup: lr = base_lr * step / warmup_steps；cosine: lr = min_lr + 0.5*(base_lr-min_lr)*(1+cos(π*progress))

## 考察点
- warmup 的必要性（Adam 早期二阶矩未稳定）
- cosine decay 的公式
- LambdaLR 的使用方式

In [ ]:
import torch
import math

class WarmupCosineScheduler:
    def __init__(self, optimizer: nn.Module, warmup_steps: int, total_steps: int, min_lr_ratio: float = 0.1) -> None:
        self.opt = optimizer
        self.warmup = warmup_steps
        self.total = total_steps
        self.min_ratio = min_lr_ratio
        self.base_lrs = [g['lr'] for g in optimizer.param_groups]
        self.step_num = 0

    def step(self) -> None:
        self.step_num += 1
        for i, g in enumerate(self.opt.param_groups):
            base = self.base_lrs[i]
            if self.step_num <= self.warmup:
                lr = base * self.step_num / self.warmup
            else:
                progress = (self.step_num - self.warmup) / (self.total - self.warmup)
                progress = min(progress, 1.0)
                min_lr = base * self.min_ratio
                lr = min_lr + 0.5 * (base - min_lr) * (1 + math.cos(math.pi * progress))
            g['lr'] = lr

In [ ]:
# 验证 lr 曲线
p = torch.tensor([1.0], requires_grad=True)
opt = torch.optim.SGD([p], lr=1.0)
sched = WarmupCosineScheduler(opt, warmup_steps=10, total_steps=100, min_lr_ratio=0.1)
lrs = []
for _ in range(100):
    sched.step()
    lrs.append(opt.param_groups[0]['lr'])
assert lrs[0] < lrs[9], "warmup 阶段应递增"
assert abs(lrs[9] - 1.0) < 0.15, "warmup 结束时 lr≈base_lr"
assert lrs[99] < 0.15, "cosine 结束时 lr≈min_lr"
assert lrs[99] > 0.05, "不应低于 min_lr"
print(f"warmup 开始 lr: {lrs[0]:.4f}")
print(f"warmup 结束 lr: {lrs[9]:.4f}")
print(f"cosine 结束 lr: {lrs[99]:.4f}")
print("✅ Warmup 递增 + Cosine 衰减验证通过")